# 04 – Room Graph
- 1 node = 1 room
- adjacency graph = rooms sharing a wall
- access graph = rooms connected through a door/window

In [1]:
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
import colorsys

print(Helper.Version())
renderer = "vscode"
GEO  = "c:/Users/etmaglari/IAAC/etmaglari_gML/Rhino_Geometry _etm.obj"
APTS = "c:/Users/etmaglari/IAAC/etmaglari_gML/Rhino_Apertures _etm.obj"

c:\Users\etmaglari\IAAC\etmaglari_gML\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The version that you are using (0.9.18) is OLDER than the latest version (0.9.20) from PyPI. Please consider upgrading to the latest version.


## 1. Load geometry + build CellComplex

In [2]:
geo = Topology.ByOBJPath(GEO)
faces = Topology.Faces(Cluster.ByTopologies(geo))
print(f"{len(faces)} faces loaded")

cc = CellComplex.ByFaces(faces, tolerance=0.01)
rooms = Topology.Cells(cc)
print(f"{len(rooms)} room(s) detected")

317 faces loaded
10 room(s) detected


## 2. Colour each room and show

In [3]:
n = len(rooms)
colors = ["#%02x%02x%02x" % tuple(int(c*255) for c in colorsys.hsv_to_rgb(i/n, 0.8, 0.9))
          for i in range(n)]

for i, room in enumerate(rooms):
    room = Topology.SetDictionary(room, Dictionary.ByKeysValues(["color", "id"], [colors[i], i]))

Topology.Show(cc,
              faceColorKey="color",
              faceOpacity=0.5,
              edgeColor="white", edgeWidth=1,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 3. Adjacency graph — connected through shared walls

In [4]:
g_adj = Graph.ByTopology(cc)
print(f"Adjacency: {len(Graph.Vertices(g_adj))} nodes, {len(Graph.Edges(g_adj))} edges")

for v in Graph.Vertices(g_adj):
    v = Topology.SetDictionary(v, Dictionary.ByKeysValues(["size","color"], [15, "red"]))
for e in Graph.Edges(g_adj):
    e = Topology.SetDictionary(e, Dictionary.ByKeysValues(["width","color"], [3, "white"]))

Topology.Show(cc, g_adj,
              faceOpacity=0.05,
              vertexSizeKey="size", vertexColorKey="color",
              edgeWidthKey="width", edgeColorKey="color",
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

Adjacency: 10 nodes, 16 edges


## 4. Load apertures (doors & windows)

In [5]:
apt = Topology.ByOBJPath(APTS)
apertures = Topology.Faces(Cluster.ByTopologies(apt))
print(f"{len(apertures)} aperture face(s) loaded")

cc_apts = Topology.AddApertures(cc, apertures)
print("Apertures added to CellComplex.")

21 aperture face(s) loaded
Apertures added to CellComplex.


## 5. Access graph — connected only through doors/windows

In [6]:
g_access = Graph.ByTopology(cc_apts, direct=False, viaSharedApertures=True)
print(f"Access: {len(Graph.Vertices(g_access))} nodes, {len(Graph.Edges(g_access))} edges")

for v in Graph.Vertices(g_access):
    v = Topology.SetDictionary(v, Dictionary.ByKeysValues(["size","color"], [15, "red"]))
for e in Graph.Edges(g_access):
    e = Topology.SetDictionary(e, Dictionary.ByKeysValues(["width","color"], [3, "#2CFF96"]))

Topology.Show(cc_apts, g_access,
              faceOpacity=0.05,
              vertexSizeKey="size", vertexColorKey="color",
              edgeWidthKey="width", edgeColorKey="color",
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

Access: 10 nodes, 0 edges
